# Tools  🔨🪚🔩⚙️⚒️🧲🧰🔧🪛🔩⚙️🔦🧭🔗⛓️🧮
Tools allow agents to 'Act' in the real world. 
Careful descriptions can help your agent discover how to use your tools.

LangChain supports many tool formats and tool sets. Here we will cover some common cases, but check the [docs](https://docs.langchain.com/oss/python/langchain/tools) for more information.

## Setup

Load and/or check for needed environmental variables

In [12]:
from dotenv import load_dotenv
from marshmallow.fields import Decimal
from pydantic.v1 import PositiveInt

from env_utils import doublecheck_env
#检查环境加载key,跳过
# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=****ywsA
LANGSMITH_API_KEY=****b63c
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


## Calculator example

In this example, the docstring and inferred arguments and argument types are used by the LLM to detetermine when and how to call the tool.

In [3]:
#导入类型注解工具(其实有很多种,比如Field等)
from typing import Literal
#导入langchain的工具装饰器
from langchain.tools import tool

#先看@tool
#将普通函数装饰为一个工具,ai能看到工具名,工具描述和参数情况
@tool
def real_number_calculator(
    #这里设置工具参数,此时operation使用类型注解工具,表示只能在这四个选项中选择,并且函数返回值为float
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    #这里的docstring会作为工具描述传给模型
    """Perform basic arithmetic operations on two real numbers."""
    print("🧮 Invoking calculator tool")
    # Perform the specified operation
    #以下就是实现计算步骤
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [4]:
from langchain.agents import create_agent
#创建agent并将上面的工具填入
agent = create_agent(
    #模型依然改为deepseek
    model="deepseek-chat",
    # model="openai:gpt-5",
    tools=[real_number_calculator],
    system_prompt="You are a helpful assistant",
)

This invokes your calculator tool.

In [5]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "what is 3.1125 * 4.1234"}]}
)
print(result["messages"][-1].content)

🧮 Invoking calculator tool
3.1125 × 4.1234 = 12.8340825


We can check the **metadata** in [LangSmith Observability](https://smith.langchain.com/public/b77bde6c-f0ad-4256-bfab-7d514ece3405/r) to see this.

The tool description can have a big impact. 
This may not  invoke your calculator tool because the inputs are integers.  (results vary from run to run)

In [6]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3 * 4"}]})
print(result["messages"][-1].content)

🧮 Invoking calculator tool
3 multiplied by 4 equals **12**.


This often does not invoke the tool though the input are real numbers. (results vary from run to run)

In [7]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3.0 * 4.0"}]})
print(result["messages"][-1].content)

🧮 Invoking calculator tool
3.0 × 4.0 = 12.0


## Adding a more detailed description
While a basic description is often sufficient, LangChain has support for enhanced descriptions. The example below uses one method: Google Style argument descriptions. Used with `parse_docstring=True`, this will parse and pass the arg descriptions to the model. You can rename the tool and change its description. This can be effective when you are sharing a standard tool but would like agent-specific instructions.

In [8]:
from typing import Literal

from langchain.tools import tool

#进阶写法
#在tool中直接指定工具名称和描述
#并且parse_docstring为True会解析docstring中的参数说明
#这里的docstring采用Google风格写法
@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Perform basic arithmetic operations on two real numbers."
        "Use this whenever you have operations on any numbers, even if they are integers."
    ),
)
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    #先描述
    #之后写上参数描述
    #具体形式如下,一般写参数名(类型):描述
    #并且可以有返回值等其他参数
    """Perform basic arithmetic operations on two real numbers.

    Args:
        a (float): The first number.
        b (float): The second number.
        operation (Literal["add", "subtract", "multiply", "divide"]):
            The arithmetic operation to perform.

            - `"add"`: Returns the sum of `a` and `b`.
            - `"subtract"`: Returns the result of `a - b`.
            - `"multiply"`: Returns the product of `a` and `b`.
            - `"divide"`: Returns the result of `a / b`. Raises an error if `b` is zero.

    Returns:
        float: The numerical result of the specified operation.

    Raises:
        ValueError: If an invalid operation is provided or division by zero is attempted.
    """
    print("🧮  Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    # model="openai:gpt-5-mini",
    model="deepseek-chat",
    tools=[real_number_calculator],
    system_prompt="You are a helpful assistant",
)

In [11]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3.0 * 4.0"}]})
print(result["messages"][-1].content)

🧮  Invoking calculator tool
3.0 × 4.0 = 12.0


Let's check our [LangSmith Observability trace](https://smith.langchain.com/public/7d65902c-bd3c-4fc6-bbd3-7c1d66566fda/r) to see the tool description.

In [12]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3 * 4"}]})
print(result["messages"][-1].content)

🧮  Invoking calculator tool
3 multiplied by 4 equals **12**.


## Try your own.
Create a tool of your own and try it here!

In [11]:

@tool
def your_tool(
    a: float, b: float, 
) -> float:
    """Perform your favorite operation

    Args:
        a (float): operator a description
        b (float): operator b description

    Returns:
        float: description
    """
    pass

In [19]:

from pydantic import BaseModel, FiniteFloat, Field, ConfigDict

#使用三层模式
#第一层类用来做输入类型约束
#第二层工具装饰
#第三层内部逻辑实现

class AbsoluteDifferenceInput(BaseModel):
    """绝对差工具的输入参数。"""
    model_config = ConfigDict(extra="forbid")
    #数据类型为有限小数
    a: FiniteFloat = Field(description="第一个数")
    b: FiniteFloat = Field(description="第二个数")

#主函数负责实际处理
def _absolute_difference(a: float, b: float,) -> float:
    """返回两个值的差的绝对值"""
    return abs(a-b)

#负责将函数打包为工具
@tool(args_schema=AbsoluteDifferenceInput)
def absolute_difference(a: float, b: float,) -> float:
    """返回两个值的差的绝对值"""
    return _absolute_difference(a, b)

In [20]:
from decimal import Decimal, ROUND_HALF_UP
from pydantic import PositiveInt, Field, ConfigDict,BaseModel
from langchain.tools import tool

class LoanMonthlyPaymentInput(BaseModel):
    """贷款月供,等额本息,年化利率用百分数,结果保留两位小数,0利率不被允许"""
    model_config = ConfigDict(extra="forbid")
    #使用Decimal防止浮点精度问题
    #gt指greater than ge指 greater than or equal
    #lt和le也指相同意义
    principal:Decimal = Field(gt=0,description="本金")
    annual_rate:Decimal = Field(gt=0,le=100,description="年化利率")
    #使用正整数PositiveInt
    months:PositiveInt = Field(description="期数")

def _loan_monthly_payment(principal:Decimal,annual_rate:Decimal,months:PositiveInt) -> Decimal:
    """计算月供,等额本息,保留两位小数"""
    #月利率
    rate = annual_rate / 1200
    # 复利因子：(1 + 月利率)^期数
    x = (1 + rate) ** months
    #等额本息月供公式
    money = (principal * rate * x) / (x - 1)
    #金额结果统一保留两位小数，使用四舍五入
    return money.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

@tool(args_schema=LoanMonthlyPaymentInput)
def loan_monthly_payment(principal:Decimal,annual_rate:Decimal,months:PositiveInt) -> Decimal:
    """计算月供,等额本息,返回两位小数的每月应还金额"""
    return _loan_monthly_payment(principal,annual_rate,months)

In [21]:
from langchain.agents import create_agent

agent = create_agent(
    model="deepseek-chat",
    # model="openai:gpt-5-mini",
    tools=[absolute_difference,loan_monthly_payment],
    system_prompt="You are a helpful assistant",
)

In [22]:

prompt="""
        你现在是一个工具测试助手。请依次测试以下两个工具：
        1. absolute_difference
        2. loan_monthly_payment

        要求：
        1. 每个工具测试 3 组正常输入和 2 组边界/异常输入。
        2. 对 absolute_difference，测试正数、负数、小数，以及非法输入。
        3. 对 loan_monthly_payment，测试常见贷款场景、较大期数、较小利率，以及业务不允许的输入。
        4. 每次调用工具后，只简洁说明你传入了什么参数、工具返回了什么结果。
        5. 如果工具抛出异常，要明确说明异常内容，不要掩盖错误。
        6. 输出要简洁、条理清楚，不要长篇解释。
"""

for step in agent.stream({"messages":[{"role": "user", "content": prompt}]},
                         stream_mode="values"):
    step["messages"][-1].pretty_print()


================================ Human Message =================================


        你现在是一个工具测试助手。请依次测试以下两个工具：
        1. absolute_difference
        2. loan_monthly_payment

        要求：
        1. 每个工具至少测试 3 组正常输入和 2 组边界/异常输入。
        2. 对 absolute_difference，测试正数、负数、小数，以及非法输入。
        3. 对 loan_monthly_payment，测试常见贷款场景、较大期数、较小利率，以及业务不允许的输入。
        4. 每次调用工具后，只简洁说明你传入了什么参数、工具返回了什么结果。
        5. 如果工具抛出异常，要明确说明异常内容，不要掩盖错误。
        6. 输出要简洁、条理清楚，不要长篇解释。

================================== Ai Message ==================================

我将依次测试这两个工具。

## 1. absolute_difference 测试

### 正常输入测试：
Tool Calls:
  absolute_difference (call_00_XLJVeDKANavfPQrhtoXQCOvk)
 Call ID: call_00_XLJVeDKANavfPQrhtoXQCOvk
  Args:
    a: 10
    b: 5
================================= Tool Message =================================
Name: absolute_difference

5.0
================================== Ai Message ==================================

传入参数：a=10, b=5
返回结果：5.0
Tool Calls:
  absolute_difference (call_0